In [1]:
#!pip install grape
from grape import Graph

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (1.26.9) or chardet (5.1.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  warnings.warn(
/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
##!pip install pykeen

In [2]:
import numpy as np
import pandas as pd
import pickle

## Notebook to create edge prediction pipeline with NP-KG (v2.1) and evaluate with reference dataset

#### Embedding methods to include: TransE, RotatE, LINE, HOPE?

In [2]:
KG_PATH = '/home/sanya/np-kg/resources/knowledge_graphs/'
DIR_PATH = 'grape-embeddings/'

In [4]:
npkg = Graph.from_csv(
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE_v2.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [5]:
#remove disconnected nodes (currently none)
npkg = npkg.remove_disconnected_nodes()

In [6]:
npkg

<div class="graph-report"><style>.graph-report li {margin: 0.5em 0 0.5em 0;}.graph-report .paragraph {text-align: justify;word-break: break-all;}.graph-report .small-columns {column-count: 4;column-gap: 2em;}.graph-report .medium-columns {column-count: 3;column-gap: 2em;}.graph-report .large-columns {column-count: 2;column-gap: 2em;}.graph-report .single-column {}@media only screen and (max-width: 600px) {.graph-report .small-columns {column-count: 1;}.graph-report .medium-columns {column-count: 1;}.graph-report .large-columns {column-count: 1;}}@media only screen and (min-width: 600px) and (max-width: 800px) {.graph-report .small-columns {column-count: 2;}.graph-report .medium-columns {column-count: 1;}.graph-report .large-columns {column-count: 1;}}@media only screen and (min-width: 800px) and (max-width: 1024px) {.graph-report .small-columns {column-count: 3;}.graph-report .medium-columns {column-count: 2;}.graph-report .large-columns {column-count: 1;}}</style><p class="paragraph">The directed multigraph has 1.09M nodes and 7.92M heterogeneous edges. The RAM requirements for the nodes and edges data structures are 131.56MB and 40.41MB respectively.</p><h3 style="margin: 1em 0 0 0;">Degree centrality</h3><p class="paragraph">The minimum node degree is 0, the maximum node degree is 21.25K, the mode degree is 2, the mean degree is 7.27 and the node degree median is 2.</p><p class="paragraph">The nodes with the highest degree centrality are <a href='http://purl.obolibrary.org/obo/UBERON_0000473' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0000473'>uberon:0000473</a> (degree 21.25K), <a href='http://amigo.geneontology.org/amigo/term/go:0005515' target='_blank' title='Go to Gene Ontology to get more informations about go:0005515'>go:0005515</a> (degree 12.33K), <a href='http://purl.obolibrary.org/obo/UBERON_0001323' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0001323'>uberon:0001323</a> (degree 11.21K), <a href='http://purl.obolibrary.org/obo/UBERON_0000007' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0000007'>uberon:0000007</a> (degree 11.21K) and <a href='http://purl.obolibrary.org/obo/UBERON_0002046' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0002046'>uberon:0002046</a> (degree 11.20K).</p><h3 style="margin: 1em 0 0 0;">Weights</h3><p class="paragraph">The minimum edge weight is 0.20001519, the maximum edge weight is 5233.8687 and the total edge weight is 2078863527.7478096. The RAM requirement for the edge weights data structure is 31.69MB.</p><h3 style="margin: 1em 0 0 0;">Edge types</h3><p class="paragraph">The graph has 355 edge types, of which the 10 most common are http:&#x2f;&#x2f;www.w3.org&#x2f;2000&#x2f;01&#x2f;rdf-schema#subclassof (1.57M edges, 19.76%), ro:0002436 (976.44K edges, 12.32%), ro:0001025 (681.38K edges, 8.60%), ro:0001015 (681.19K edges, 8.60%), ro:0002201 (433.59K edges, 5.47%), ro:0002200 (433.59K edges, 5.47%), ro:0000057 (381.14K edges, 4.81%), ro:0000056 (381.13K edges, 4.81%), ro:0003302 (305.94K edges, 3.86%) and ro:0002160 (287.23K edges, 3.63%). The RAM requirement for the edge types data structure is 31.73MB.</p><h4 style="margin: 1em 0 0 0;">Singleton edge types</h4><p class="paragraph">Singleton edge types are edge types that are assigned exclusively to a single edge, making the edge type relatively meaningless, as it adds no more information than the name of edge itself. The graph contains 41 edges with singleton edge types, which are ro:0002482, exo#interacts:with:an:exposure:stressor:via, uberon&#x2f;core#posteriorly:connected:to, ro:0002481, ro:0002493, mondo#disease:has:basis:in:accumulation:of, uberon&#x2f;core#evolved:from, ro:0002497, ro:0002336 and mf#manifestationof, plus other 31 edges with singleton edge types. </p><h3 style="margin: 1em 

In [8]:
npkg.get_number_of_connected_components()

(1, 1090172, 1090172)

In [5]:
npkg.enable()

In [12]:
##test loading graph with node types
npkg_new = Graph.from_csv(
    node_path=KG_PATH+'nodeTypes_CURIE_v2.0.0.tsv',
    node_list_node_types_column_number=1,
    nodes_column_number=0,
    node_list_separator='\t',
    node_list_header=True,
    edge_path=KG_PATH+'NP-KG_BETWEENNESS-WEIGHTED-PK-CURIE_v2.0.0.tsv',
    edge_list_separator='\t',
    edge_list_header=True,
    edge_list_edge_types_column_number=1,
    sources_column_number=0,
    destinations_column_number=2,
    weights_column_number=3,
    directed=True,
    verbose=True
)

In [13]:
npkg_new

<div class="graph-report"><style>.graph-report li {margin: 0.5em 0 0.5em 0;}.graph-report .paragraph {text-align: justify;word-break: break-all;}.graph-report .small-columns {column-count: 4;column-gap: 2em;}.graph-report .medium-columns {column-count: 3;column-gap: 2em;}.graph-report .large-columns {column-count: 2;column-gap: 2em;}.graph-report .single-column {}@media only screen and (max-width: 600px) {.graph-report .small-columns {column-count: 1;}.graph-report .medium-columns {column-count: 1;}.graph-report .large-columns {column-count: 1;}}@media only screen and (min-width: 600px) and (max-width: 800px) {.graph-report .small-columns {column-count: 2;}.graph-report .medium-columns {column-count: 1;}.graph-report .large-columns {column-count: 1;}}@media only screen and (min-width: 800px) and (max-width: 1024px) {.graph-report .small-columns {column-count: 3;}.graph-report .medium-columns {column-count: 2;}.graph-report .large-columns {column-count: 1;}}</style><p class="paragraph">The directed multigraph has 1.09M heterogeneous nodes and 7.92M heterogeneous edges. The RAM requirements for the nodes and edges data structures are 131.58MB and 40.41MB respectively.</p><h3 style="margin: 1em 0 0 0;">Degree centrality</h3><p class="paragraph">The minimum node degree is 0, the maximum node degree is 21.25K, the mode degree is 2, the mean degree is 7.27 and the node degree median is 2.</p><p class="paragraph">The nodes with the highest degree centrality are <a href='http://purl.obolibrary.org/obo/UBERON_0000473' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0000473'>uberon:0000473</a> (degree 21.25K and node type anatomy), <a href='http://amigo.geneontology.org/amigo/term/go:0005515' target='_blank' title='Go to Gene Ontology to get more informations about go:0005515'>go:0005515</a> (degree 12.33K and node type process), <a href='http://purl.obolibrary.org/obo/UBERON_0001323' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0001323'>uberon:0001323</a> (degree 11.21K and node type anatomy), <a href='http://purl.obolibrary.org/obo/UBERON_0000007' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0000007'>uberon:0000007</a> (degree 11.21K and node type anatomy) and <a href='http://purl.obolibrary.org/obo/UBERON_0002046' target='_blank' title='Go to Uberon multi-species anatomy ontology to get more informations about uberon:0002046'>uberon:0002046</a> (degree 11.20K and node type anatomy).</p><h3 style="margin: 1em 0 0 0;">Weights</h3><p class="paragraph">The minimum edge weight is 0.20001519, the maximum edge weight is 5233.8687 and the total edge weight is 2078863437.7478094. The RAM requirement for the edge weights data structure is 31.69MB.</p><h3 style="margin: 1em 0 0 0;">Node types</h3><p class="paragraph">The graph has 36 node types, of which the 10 most common are variant (236.16K nodes, 21.66%), protein (227.34K nodes, 20.85%), sequence (197.39K nodes, 18.10%), chemical (164.53K nodes, 15.09%), process (43.25K nodes, 3.97%), cell_line (39.08K nodes, 3.58%), gene (31.65K nodes, 2.90%), disease (24.09K nodes, 2.21%), other (18.81K nodes, 1.72%) and phenotype (17.14K nodes, 1.57%). The RAM requirement for the node types data structure is 78.52MB.</p><h3 style="margin: 1em 0 0 0;">Edge types</h3><p class="paragraph">The graph has 355 edge types, of which the 10 most common are http:&#x2f;&#x2f;www.w3.org&#x2f;2000&#x2f;01&#x2f;rdf-schema#subclassof (1.57M edges, 19.76%), ro:0002436 (976.44K edges, 12.32%), ro:0001025 (681.38K edges, 8.60%), ro:0001015 (681.19K edges, 8.60%), ro:0002200 (433.59K edges, 5.47%), ro:0002201 (433.59K edges, 5.47%), ro:0000057 (381.14K edges, 4.81%), ro:0000056 (381.13K edges, 4.81%), ro:0003302 (305.94K edges, 3.86%) and ro:0002160 (287.23K edges, 3.63%). The RAM requirement for the edge types data structure is 31.73MB.</p><h4 style

## Explore graph structure, weights, and nodes

See npkg-workspace/weighting for centrality measures

In [10]:
##see above report for weights and degree min, max, mean, median, mode
npkg.get_top_k_central_node_names(k=10)

['uberon:0000473',
 'go:0005515',
 'uberon:0001323',
 'uberon:0000007',
 'uberon:0002046',
 'uberon:0002367',
 'uberon:0003889',
 'uberon:0000992',
 'uberon:0000458',
 'uberon:0012249']

In [11]:
npkg.get_available_edge_metrics_names()


['Adamic Adar',
 'Jaccard Coefficient',
 'Resource allocation index',
 'Preferential attachment']

In [12]:
npkg.get_average_clustering_coefficient()


0.03551581589784823

In [13]:
npkg.get_clustering_coefficient()

38718.34804898899

In [14]:
npkg.get_density()

5.825633741290142e-06

## Generate embeddings

### TransE

In [6]:
from grape.embedders import TransEEnsmallen

In [21]:
#Defaults: embedding_size=100, epochs=100, relu_bias=1.0, learning_rate=0.05, learning_rate_decay=0.9, random_state=42
#Time taken on CPU: 8 minutes
transe_embed_model = TransEEnsmallen(
    embedding_size=100,
    epochs=100,
    node_embedding_path=DIR_PATH+'npkg-transE-node-embedding-default.txt',
    edge_type_embedding_path=DIR_PATH+'npkg-transE-edge-type-embedding-default.txt',
    random_state=42,
    verbose=True,
    enable_cache=True
)
transe_embedding = transe_embed_model.fit_transform(npkg)

In [34]:
transe_embed_model.parameters()

{'random_state': 42,
 'embedding_size': 100,
 'relu_bias': 1.0,
 'epochs': 100,
 'learning_rate': 0.05,
 'learning_rate_decay': 0.9,
 'node_embedding_path': 'grape-embeddings/npkg-transE-node-embedding-default.txt',
 'verbose': True,
 'dtype': 'f32',
 'edge_type_embedding_path': 'grape-embeddings/npkg-transE-edge-type-embedding-default.txt'}

In [22]:
##save embeddings (if no path provided above)
import pickle
transe_embedding_dict = transe_embedding.dump()
with open(DIR_PATH+'npkg_transE_embedding_dict_default.pickle', 'wb') as file_trans:
    pickle.dump(transe_embedding_dict, file_trans)

In [7]:
#load embeddings
from grape import EmbeddingResult
with open(DIR_PATH+'npkg_transE_embedding_dict_default.pickle', 'rb') as file_i:
    transe_embedding_dict = pickle.load(file_i)
transe_embedding = EmbeddingResult.load(transe_embedding_dict)

In [8]:
##take a look at the node embedding
#index of dataframe are the node URIs from the graph
transe_embedding.get_node_embedding_from_index(0)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
ensembl:ENST00000669649,0.133012,-0.039734,-0.017600,0.074753,0.089565,-0.129705,-0.061305,-0.112825,-0.035546,0.039217,...,0.091381,-0.055465,-0.049071,0.015248,0.085441,0.127182,-0.048844,-0.007128,0.117262,0.134768
uberon:0006566,-0.042953,0.119542,0.106486,-0.095515,-0.072714,0.052780,0.075805,0.058281,0.089060,-0.130521,...,-0.048847,0.095971,0.100481,-0.161149,-0.065478,-0.082184,0.092494,-0.150797,-0.061185,-0.062067
ensembl:ENST00000710852,0.129897,-0.055989,-0.048424,0.068339,0.081475,-0.126741,-0.086929,-0.112643,-0.063531,0.071894,...,0.110342,-0.075432,-0.068375,0.034195,0.100543,0.146299,-0.071475,0.030592,0.111794,0.122829
so:0001877,-0.032624,-0.112300,-0.091139,-0.074079,-0.072230,0.035402,-0.120046,0.045256,-0.112124,0.127538,...,0.154696,-0.133258,-0.120145,0.086671,0.140970,0.131788,-0.126846,0.084415,-0.045741,-0.062484
uberon:0002046,-0.043733,0.120863,0.109191,-0.096344,-0.073366,0.053372,0.077542,0.059337,0.091038,-0.131430,...,-0.049722,0.097291,0.101299,-0.162655,-0.066260,-0.083180,0.093742,-0.153220,-0.061787,-0.063820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
pr:p51984,-0.071030,-0.122624,-0.046186,-0.046063,-0.066687,0.062966,-0.114123,0.046304,-0.103504,0.140603,...,0.097197,-0.136954,-0.110243,0.113062,0.138561,0.285851,-0.115542,0.087326,-0.049446,-0.079660
pr:q3iz91,-0.063238,-0.118321,-0.114408,-0.095154,-0.072189,0.095164,-0.114518,0.090444,-0.091321,0.123619,...,0.117939,-0.113842,-0.124325,0.061563,0.129074,0.135218,-0.128681,0.108919,-0.072243,-0.073731
go:0034019,-0.132350,-0.073959,-0.074117,-0.136843,-0.109497,0.114793,-0.116247,0.115912,-0.080128,0.071386,...,0.108645,-0.084689,-0.093388,0.059807,0.095091,0.099150,-0.081326,0.062787,-0.104859,-0.107982
pr:p83965,-0.041061,-0.127165,-0.111282,-0.081521,-0.061617,0.068471,-0.141537,0.057336,-0.118830,0.099643,...,0.128686,-0.113616,-0.147926,0.095228,0.133988,0.160273,-0.103372,0.095524,-0.116441,-0.019295


In [9]:
##take a look at the edge embedding (generated for the 4 different edge types in the graph)
transe_embedding.get_edge_type_embedding_from_index(0)

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
ro:0001025,-6.636734,7.613238,8.224732,-7.443454,-8.477786,5.672297,9.893656,6.597908,9.296768,-5.454515,...,-9.752181,7.710365,9.926104,-6.169213,-10.322266,-5.666488,8.394701,-6.800806,-6.763617,-6.817780
ro:0001015,7.695096,-6.392290,-7.731688,8.045654,9.637430,-6.672614,-8.348529,-7.590189,-8.243292,4.813333,...,8.375585,-6.841699,-7.863305,5.166222,8.724146,5.584220,-7.568592,5.699937,8.164729,8.129503
http://www.w3.org/2000/01/rdf-schema#subclassof,-4.857944,-4.488316,-5.306038,-5.061757,-5.226358,4.625896,-5.259227,4.903893,-5.411017,3.888401,...,5.417539,-4.956500,-4.908848,4.176100,5.219866,2.498688,-5.245125,4.570869,-4.770346,-4.184330
bfo:0000050,-3.382227,2.835286,4.321743,-3.384536,-3.567644,3.448605,4.351221,3.640927,4.360774,2.665185,...,-4.290301,3.729779,3.589310,2.424194,-4.167218,-3.289349,3.833445,-2.947739,-3.393661,-3.561412
ro:0000056,6.205667,-5.067720,-5.573077,6.795090,-6.602760,-6.291440,7.374416,-6.419090,-5.992821,4.678592,...,-7.274004,-5.312018,6.786849,4.380886,-7.060073,5.181231,-5.605486,4.653585,6.386588,-5.182482
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ro:0002509,-0.314047,-0.218692,-0.195120,0.088319,-0.049960,-0.076689,0.031821,0.195661,-0.197387,-0.142221,...,-0.037379,-0.132735,-0.183673,0.325302,-0.139012,0.058360,-0.026303,-0.169888,-0.277753,-0.001862
so#transcribed:from,-0.245226,-0.321950,-0.254210,-0.101990,-0.253767,-0.131768,-0.029788,0.154678,0.003670,0.315482,...,0.191094,-0.289768,-0.192100,-0.073771,-0.192126,-0.347457,-0.078338,0.048236,0.068350,0.047734
foodon:00001301,0.095848,0.106489,-0.277093,0.131017,-0.212231,0.102525,-0.211474,0.100451,0.130998,-0.058926,...,-0.024576,0.050765,0.061389,0.215195,0.026648,0.231535,0.024710,0.175974,-0.188626,0.147543
ncit:r81,-0.051298,-0.010116,-0.025805,0.086762,-0.147431,-0.040595,0.101275,-0.239957,-0.021895,-0.092163,...,-0.280338,0.132704,-0.147428,-0.070553,0.060339,-0.186369,0.235242,-0.110890,0.289624,-0.145919


### RotatE - INCOMPLETE

Using PyKEEN embedder wrappers, see pykeen notebook for experimentation with PyKEEN directly. Evaluation is not imported, only the models, so will also use PyKEEN directly for embeddings 
generation and evaluation.

In [27]:
from grape.embedders import RotatEPyKEEN

In [30]:
#Default: embedding_size=100, relation_dim=30, epochs=100, random_state=42, training_loop=stochastic local closed world assumption, scoring_fct_norm (gamma) = 2, batch_size=2**10
transr_embed_model = RotatEPyKEEN()

/home/sanya/.conda/envs/kg-embed/lib/python3.8/site-packages/torch/cuda/__init__.py:141: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [ ]:
##model uses default TransR, TransD, RotatE hyperparameters from original papers
#can be modified using model args and kwargs but hyperparameter optimization needs to be implemented separately if using GRAPE

In [ ]:
##See https://github.com/monarch-initiative/embiggen/blob/95b18f96ad3a613e48509a1bef21804f6329f728/embiggen/embedders/pykeen_embedders/entity_relation_embedding_model_pykeen.py
# for how to convert PyKEEN embeddings to GRAPE embedding dataframes to use below (possibly)

## Hyperparameter optimization - TODO

In [ ]:
embedding_sizes = [50, 100, 200]
learning_rates = [0.01, 0.1, 0.05]
relu_bias = [1, 2, 10]

In [ ]:
best_performance = -np.inf
best_hyperparameters = None
best_embedding_result = None

In [ ]:
for embedding_size in embedding_sizes:
    for learning_rate in learning_rates:
        for bias in relu_bias:
            print(f'Embedding size: {embedding_size}, learning rate: {learning_rate}, bias: {bias}')
            embed_model = TransEEnsmallen(
                embedding_size=embedding_size,
                epochs=100,
                relu_bias=bias,
                learning_rate=learning_rate,
                node_embedding_path=EMBEDDING_PATH+f'npkg_transE_node_{embedding_size}_{learning_rate}_{bias}.csv',
                edge_type_embedding_path=EMBEDDING_PATH+f'npkg_transE_edge_{embedding_size}_{learning_rate}_{bias}.csv',
                random_state=42,
                verbose=True,
                enable_cache=True
            )
            embedding_result = embed_model.fit(npkg)
            '''
            performance = evaluate_embeddings(embedding_result) -- instrinsic evaluation with MRR??

            if performance > best_performance:
                best_performance = performance
                best_hyperparameters = (embedding_size, learning_rate, bias)
                best_embedding_result = embedding_result'''

## Holdouts - instrinsic evaluation of embeddings

See https://github.com/AnacletoLAB/grape/blob/main/tutorials/Graph_holdouts_using_GRAPE.ipynb

Does not use edge prediction evaluation pipeline

In [10]:
#random holdout -- probably not the best - use connected instead
train, test = npkg.random_holdout(
    train_size=0.8,
    random_state=42,
    include_all_edge_types=True,
    #edge_types = [optional list of strings if restricting holdout/evaluation to specific edge types]
    #optional integer value that specifies the minimum number of overlaps required for an edge to be included in the validation set. This parameter is only applicable to multigraphs.
    min_number_overlaps=1,
    verbose=True
)

In [12]:
print('Train holdout:')
print('Number of edges: ', train.get_number_of_edges())
print('Number of nodes: ', train.get_number_of_nodes())
print('Edge types: ', train.get_number_of_edge_types())

Train holdout:
Number of edges:  6537574
Number of nodes:  1090172
Edge types:  355


In [13]:
print('Test holdout:')
print('Number of edges: ', test.get_number_of_edges())
print('Number of nodes: ', test.get_number_of_nodes())
print('Edge types: ', test.get_number_of_edge_types())

Test holdout:
Number of edges:  1385007
Number of nodes:  1090172
Edge types:  355


In [ ]:
##use the split for edge prediction to evaluate embeddings performance on existing graph
train_embedding = TransEEnsmallen(
    embedding_size=100,
    epochs=100,
    node_embedding_path=DIR_PATH+'npkg-train-transE-node-embedding-default.txt',
    edge_type_embedding_path=DIR_PATH+'npkg-train-transE-edge-type-embedding-default.txt',
    random_state=42,
    verbose=True,
    enable_cache=True
).fit_transform(train)

In [ ]:
'''Params:https://github.com/monarch-initiative/embiggen/blob/95b18f96ad3a613e48509a1bef21804f6329f728/embiggen/edge_prediction/edge_prediction_sklearn/mlp_edge_prediction.py
- hidden_layer_sizes=(100,)
- activation='relu'
- solver='adam'
- alpha=0.0001
- ....
- edge_embedding_methods: see Notion (default: 'concatenate')
- use_edge_metrics=False
'''
mlp_model = MLPEdgePrediction(

)
'''Params: https://github.com/monarch-initiative/embiggen/blob/95b18f96ad3a613e48509a1bef21804f6329f728/embiggen/edge_prediction/sklearn_like_edge_prediction_adapter.py#L261
- graph
- node_features - node embeddings
- node_type_features
- edge_type_features
- edge_features
'''
mlp_model.fit()

In [ ]:
model.dump(path='json or pkl')

In [ ]:
%%time
# A perfect model should correctly predict the existance
# of all of these edges.
model.predict_proba(
    graph=test,
    node_features=train_embedding.get_node_embedding_from_index,
    return_predictions_dataframe=True
)

In [ ]:
%%time
# A perfect model should correctly predict the non-existance
# of all of these edges.
model.predict_proba(
    graph=graph.sample_negative_graph(number_of_negative_samples=test.get_number_of_edges()),
    node_features=train_embedding.get_node_embedding_from_index,
    return_predictions_dataframe=True
)

If edge embeddings are not available (like in random walk based methods), the below operations are applied to the node embeddings to get edge embeddings -
* hadamard: element-wise product
* sum: element wise sum
* average: element wise mean
* L1: element wise subtraction
* AbsoluteL1: element wise subtraction in absolute value
* L2: element-wise squared root of squared subtraction
* concatenate: concatenate source and destination vectors
* min: element wise min
* max: element wise max
* L2Distance: vector wise L2 distance --- gives scalar
* Costine similarity: vector wise cosine similarity -- gives scalar

In [ ]:
##Perceptron for edge prediction - https://github.com/AnacletoLAB/grape/blob/main/tutorials/Edge%20Predictions%20on%20STRING%20Homo%20Sapiens%20using%20a%20Perceptron.ipynb
help(PerceptronEdgePrediction)

## Edge Prediction Pipeline with Reference Data

In [14]:
from grape.edge_prediction import edge_prediction_evaluation
from grape.edge_prediction import PerceptronEdgePrediction, MLPEdgePrediction, GNNEdgePrediction, GraphSAGEEdgePrediction, RandomForestEdgePrediction

In [12]:
import pandas as pd
from tqdm.auto import tqdm, trange

In [ ]:
##choose classifier
classifier_options = ['perceptron', 'random_forest', 'logistic_regression', 'multilayerperceptron', 'gnn', 'graphsage', 'lightgbm', 'xgboost']

In [ ]:
##classifier implementation
#edge embedding methods: - CosineSimilarity,EuclideanDistance,Concatenate,Hadamard,L1,L2,Add,Sub,Maximum,Minimum,
#https://github.com/AnacletoLAB/grape/blob/main/tutorials/Edge%20Predictions%20on%20STRING%20Homo%20Sapiens%20using%20a%20Perceptron.ipynb
classifier = 'perceptron'
#check params from class
model = PerceptronEdgePrediction(
    edge_features=None,
    number_of_edges_per_mini_batch=32,
    edge_embeddings=''
)
##check params from model.fit method
model.fit(
    graph=train,
    node_features=embedding,
    edge_features=embedding
)

In [ ]:
#same params as fit likely
model.predict_proba(
    graph=test,
    node_features=embedding,
    edge_features=embedding,
    return_predictions_dataframe=True
)

In [ ]:
##test on negative samples/edges
model.predict_proba(
    graph=graph.sample_negative_graph(number_of_negative_samples=test.get_number_of_edges()),
    node_features=embedding,
    edge_features=embedding,
    return_predictions_dataframe=True
)

In [ ]:
model.plot()

In [ ]:
##Using weighted vs unweighted walks in random walk based methods
def graph_callback(graph):
    """Returns standardized graph."""
    if graph.get_name() == "MP":
        return graph
    graph = graph.filter_from_names(max_edge_weight=<>)
    graph.enable()
    return graph

## Edge prediction evaluation pipeline

In [5]:
from grape.edge_prediction import edge_prediction_evaluation
##a lot of custom options - read before implementing
help(edge_prediction_evaluation)

Help on function edge_prediction_evaluation in module embiggen.edge_prediction.edge_prediction_evaluation:

edge_prediction_evaluation(holdouts_kwargs: Dict[str, Any], graphs: Union[str, ensmallen.Graph, List[ensmallen.Graph], List[str]], models: Union[Type[embiggen.edge_prediction.edge_prediction_model.AbstractEdgePredictionModel], List[Type[embiggen.edge_prediction.edge_prediction_model.AbstractEdgePredictionModel]]], evaluation_schema: str = 'Connected Monte Carlo', node_features: Union[str, pandas.core.frame.DataFrame, numpy.ndarray, Type[embiggen.utils.abstract_models.abstract_embedding_model.AbstractEmbeddingModel], List[Union[str, pandas.core.frame.DataFrame, numpy.ndarray, Type[embiggen.utils.abstract_models.abstract_embedding_model.AbstractEmbeddingModel]]], NoneType] = None, node_type_features: Union[str, pandas.core.frame.DataFrame, numpy.ndarray, Type[embiggen.utils.abstract_models.abstract_embedding_model.AbstractEmbeddingModel], List[Union[str, pandas.core.frame.DataFrame

In [ ]:
results = pd.concat([
    edge_prediction_evaluation(
        holdout_kwargs=dict(train_size=0.8),
        graphs=npkg,
        #if filtering graph by weight
        #graph_callback=graph_callback,
        models=[
            PerceptronEdgePrediction(edge_features=None, edge_embeddings='Hamdard')
        ],
        number_of_holdouts=10,
        node_features=EmbeddingMethod(##params need to be set based on embedding methods below
            epochs=10,
            window_size=5,
            iterations=2,
        ),
        smoke_test=False,
        enable_cache=True,
    )
    for EmbeddingMethod in tqdm((
        TransEEnsmallen
    ), desc="Embedding methods")
])

In [ ]:
results.to_csv()

In [ ]:
edge_prediction_results = pd.read_csv('edge_prediction_results.csv')
#get top model
top_model = edge_prediction_results[edge_prediction_results['evaluation_mode']=='test'].sort_values('matthews_correlation_coefficient', ascending=False).iloc[0]

In [ ]:
#extract graph feature params
feature_params = top_model[[col for col in top_model.index.values if 'features_parameters' in col]]
feature_params = {literal_eval(k)[1]: feature_params.loc[k] for k in feature_params.index.values}
#extract the prediction model params
model_params = top_model[[col for col in top_model.index.values if 'model_parameters' in col]]
model_params = {literal_eval(k)[1]: model_params.loc[k] for k in model_params.index.values}
#instantiate the embedding model
EmbedModel = top_model.loc['embedding_model']
EmbedModel = getattr(import_module('grape.embedders'), EmbedModel)

In [ ]:
embedder = EmbedModel(**feature_params)


In [ ]:
predmodel = EdgeClassifiers[top_model.loc['model_name']](**model_params)

In [ ]:
embedding = embedder.fit_transform(graph)
predmodel.fit(embedding)

## Visualization

In [ ]:
from grape.visualizations import GraphVisualizer

In [ ]:
GraphVisualizer(graph).fit_and_plot_all(embedding, show_letters=False)

In [15]:
help(PerceptronEdgePrediction)

Help on class PerceptronEdgePrediction in module embiggen.edge_prediction.edge_prediction_ensmallen.perceptron:

class PerceptronEdgePrediction(embiggen.edge_prediction.edge_prediction_model.AbstractEdgePredictionModel)
 |  PerceptronEdgePrediction(edge_features: Union[str, List[str], NoneType] = 'JaccardCoefficient', edge_embeddings: Union[str, List[str], NoneType] = None, cooccurrence_iterations: int = 100, cooccurrence_window_size: int = 10, number_of_epochs: int = 1000, number_of_edges_per_mini_batch: int = 4096, learning_rate: float = 0.001, first_order_decay_factor: float = 0.9, second_order_decay_factor: float = 0.999, avoid_false_negatives: bool = False, use_scale_free_distribution: bool = True, random_state: int = 42, verbose: bool = True)
 |  
 |  Perceptron model for edge prediction.
 |  
 |  Method resolution order:
 |      PerceptronEdgePrediction
 |      embiggen.edge_prediction.edge_prediction_model.AbstractEdgePredictionModel
 |      embiggen.utils.abstract_models.abstr